In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

In [ ]:
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

agents_config = configs['agents']
tasks_config = configs['tasks']

In [3]:
from crewai_tools import FileReadTool
csv_tool = FileReadTool(file_path='./support_tickets_data.csv')

## Creating Agents, Tasks and Crew

In [4]:
# Creating Agents
suggestion_generation_agent = Agent(
  config=agents_config['suggestion_generation_agent'],
  tools=[csv_tool]
)

reporting_agent = Agent(
  config=agents_config['reporting_agent'],
  tools=[csv_tool]
)

chart_generation_agent = Agent(
  config=agents_config['chart_generation_agent'],
  allow_code_execution=False
)

# Creating Tasks
suggestion_generation = Task(
  config=tasks_config['suggestion_generation'],
  agent=suggestion_generation_agent
)

table_generation = Task(
  config=tasks_config['table_generation'],
  agent=reporting_agent
)

chart_generation = Task(
  config=tasks_config['chart_generation'],
  agent=chart_generation_agent
)

final_report_assembly = Task(
  config=tasks_config['final_report_assembly'],
  agent=reporting_agent,
  context=[suggestion_generation, table_generation, chart_generation]
)


# Creating Crew
support_report_crew = Crew(
  agents=[
    suggestion_generation_agent,
    reporting_agent,
    chart_generation_agent
  ],
  tasks=[
    suggestion_generation,
    table_generation,
    chart_generation,
    final_report_assembly
  ],
  verbose=True
)


## Testing our Crew

In [5]:
support_report_crew.test(n_iterations=1, openai_model_name='gpt-4o')

2026-08-10 03:20:04,099 - 140699976911744 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: Suggestion Engine
## Task: Generate actionable suggestions for resolving each classified support ticket. The suggestions should be based on: - Issue Type: Tailor suggestions to the specific type of issue reported. - Historical Data: Use historical data such as resolution_time_minutes and
  satisfaction_rating to inform the suggestions.
- Customer Feedback: Incorporate insights from customer_comments to
  customize the suggestions further.

The goal is to provide clear, actionable steps that the support team can take to resolve each issue efficiently and effectively.



# Agent: Suggestion Engine
## Using tool: Read a file's content
## Tool Input: 
"{\"file_path\": \"./support_tickets_data.csv\"}"
## Tool Output: 
ticket_id,customer_id,issue_type,issue_description,priority,date_submitted,response_time_minutes,resolution_time_minutes,satisfaction_rating,customer_comments,agent_id,resolved
T0001,C0511,API Issue,I'm pleased with how my issue was handled. Thanks!,High,2023-03-25,24



# Agent: Suggestion Engine
## Final Answer: 
1. **API Issue - Ticket T0001**  
   - Action: Continue to utilize efficient communication protocols to prevent slowdowns.   
   - Suggestion: Ensure agents follow up on recurring API issues based on previous feedback that hints at frequent occurrences. Improve resolution speed by using pre-defined scripts for common API issues.

2. **Login Issue - Ticket T0002**  
   - Action: Investigate any system outages during the time the customer attempted to log in.  
   - Suggestion: Consistently monitor system status and provide alerts to customers about any ongoing issues. Follow the escalation path to quickly resolve persisting login issues.

3. **Report Generation - Ticket T0003**  
   - Action: Enhance the training for agents dealing with report generation.  
   - Suggestion: Develop a fast-track troubleshooting guide specifically for report generation issues to improve first-call resolutions.

4. **Data Import - Ticket T0004**  
   - Action:



# Agent: Report Generator
## Final Answer: 
**Issue Classification Results:**
| Issue Type        | Frequency | Priority Levels |
|-------------------|-----------|------------------|
| API Issue         | 5         | High (3), Medium (2) |
| Login Issue       | 8         | Critical (2), High (5), Low (1) |
| Report Generation  | 6         | High (2), Medium (2), Low (2) |
| Data Import       | 12        | High (5), Medium (5), Low (2) |
| Feature Request   | 8         | Critical (2), High (4), Medium (2) |
| Billing Issue     | 12        | Critical (3), High (5), Medium (4) |
| UI Bug            | 7         | High (3), Medium (3), Critical (1) |

**Agent Performance:**
| Agent ID | Total Resolved | Total Tickets | Avg Resolution Time (mins) | Avg Satisfaction Rating |
|----------|----------------|---------------|-----------------------------|--------------------------|
| A001     | 7              | 12            | 843.33                      | 2.67                     |
| A002     | 



# Agent: Report Generator
## Final Answer: 
**Support Ticket Summary Report**
---

### **1. Issue Classification Results:**
| Issue Type        | Frequency | Priority Levels                 |
|-------------------|-----------|----------------------------------|
| API Issue         | 5         | High (3), Medium (2)            |
| Login Issue       | 8         | Critical (2), High (5), Low (1) |
| Report Generation  | 6         | High (2), Medium (2), Low (2)   |
| Data Import       | 12        | High (5), Medium (5), Low (2)   |
| Feature Request   | 8         | Critical (2), High (4), Medium (2) |
| Billing Issue     | 12        | Critical (3), High (5), Medium (4) |
| UI Bug            | 7         | High (3), Medium (3), Critical (1) |

![Issue Distribution](./issue_distribution.png)  
![Ticket Breakdown by Priority Level](./priority_levels.png)

### **2. Agent Performance:**
| Agent ID | Total Resolved | Total Tickets | Avg Resolution Time (mins) | Avg Satisfaction Rating |
|------

                             Tasks Scores                              
                        (1-10 Higher is better)                        
┏━━━━━━━━━━━━━━━━━━━━┯━━━━━━━┯━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━┯━━┯━━┓
┃ Tasks/Crew/Agents  │ Run 1 │ Avg. Total │ Agents              │  │  ┃
┠────────────────────┼───────┼────────────┼─────────────────────┼──┼──┨
┃ Task 1             │  9.0  │    9.0     │ - Suggestion Engine │  │  ┃
┃                    │       │            │                     │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Task 2             │  8.5  │    8.5     │ - Report Generator  │  │  ┃
┃                    │       │            │                     │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Task 3             │  9.5  │    9.5     │ - Chart Specialist  │  │  ┃
┃                    │       │            │                     │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Task 4             │  8.5  │    8.5     │ - Report Generator  │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Crew               │ 8.88  │    8.9     │                     │  │  ┃
┃ Execution Time (s) │   3   │     3      │                     │  │  ┃
┗━━━━━━━━━━━━━━━━━━━━┷━━━━━━━┷━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━┷━━┷━━┛

## Training your crew and agents

In [6]:
support_report_crew.train(n_iterations=1, filename='training.pkl')

2026-08-10 03:20:08,245 - 140699976911744 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: Suggestion Engine
## Task: Generate actionable suggestions for resolving each classified support ticket. The suggestions should be based on: - Issue Type: Tailor suggestions to the specific type of issue reported. - Historical Data: Use historical data such as resolution_time_minutes and
  satisfaction_rating to inform the suggestions.
- Customer Feedback: Incorporate insights from customer_comments to
  customize the suggestions further.

The goal is to provide clear, actionable steps that the support team can take to resolve each issue efficiently and effectively.



# Agent: Suggestion Engine
## Using tool: Read a file's content
## Tool Input: 
"{\"file_path\": \"./support_tickets_data.csv\"}"
## Tool Output: 
ticket_id,customer_id,issue_type,issue_description,priority,date_submitted,response_time_minutes,resolution_time_minutes,satisfaction_rating,customer_comments,agent_id,resolved
T0001,C0511,API Issue,I'm pleased with how my issue was handled. Thanks!,High,2023-03-25,24



# Agent: Suggestion Engine
## Final Answer: 
1. **API Issue - Ticket T0001**  
   - Action: Continue to utilize efficient communication protocols to prevent slowdowns.   
   - Suggestion: Ensure agents follow up on recurring API issues based on previous feedback that hints at frequent occurrences. Improve resolution speed by using pre-defined scripts for common API issues.

2. **Login Issue - Ticket T0002**  
   - Action: Investigate any system outages during the time the customer attempted to log in.  
   - Suggestion: Consistently monitor system status and provide alerts to customers about any ongoing issues. Follow the escalation path to quickly resolve persisting login issues.

3. **Report Generation - Ticket T0003**  
   - Action: Enhance the training for agents dealing with report generation.  
   - Suggestion: Develop a fast-track troubleshooting guide specifically for report generation issues to improve first-call resolutions.

4. **Data Import - Ticket T0004**  
   - Action:

I want you to do better suggest.
 
Processing training feedback.



2026-08-10 03:22:04,509 - 140699976911744 - llm.py-llm:309 - ERROR: LiteLLM call failed: litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'exceeded quota for this month'}}




LiteLLM.Info: If you need to debug this error, use `litellm.set_verbose=True'.

 Error during LLM call: litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'exceeded quota for this month'}}
 
[2026-08-10 03:22:04][ERROR]: Training failed: litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'exceeded quota for this month'}}


RateLimitError: litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'exceeded quota for this month'}}

In [7]:
support_report_crew.test(n_iterations=1, openai_model_name='gpt-4o')

2026-08-10 03:22:44,803 - 140699976911744 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: Suggestion Engine
## Task: Generate actionable suggestions for resolving each classified support ticket. The suggestions should be based on: - Issue Type: Tailor suggestions to the specific type of issue reported. - Historical Data: Use historical data such as resolution_time_minutes and
  satisfaction_rating to inform the suggestions.
- Customer Feedback: Incorporate insights from customer_comments to
  customize the suggestions further.

The goal is to provide clear, actionable steps that the support team can take to resolve each issue efficiently and effectively.



# Agent: Suggestion Engine
## Using tool: Read a file's content
## Tool Input: 
"{\"file_path\": \"./support_tickets_data.csv\"}"
## Tool Output: 
ticket_id,customer_id,issue_type,issue_description,priority,date_submitted,response_time_minutes,resolution_time_minutes,satisfaction_rating,customer_comments,agent_id,resolved
T0001,C0511,API Issue,I'm pleased with how my issue was handled. Thanks!,High,2023-03-25,24



# Agent: Suggestion Engine
## Final Answer: 
1. **API Issue - Ticket T0001**  
   - Action: Continue to utilize efficient communication protocols to prevent slowdowns.   
   - Suggestion: Ensure agents follow up on recurring API issues based on previous feedback that hints at frequent occurrences. Improve resolution speed by using pre-defined scripts for common API issues.

2. **Login Issue - Ticket T0002**  
   - Action: Investigate any system outages during the time the customer attempted to log in.  
   - Suggestion: Consistently monitor system status and provide alerts to customers about any ongoing issues. Follow the escalation path to quickly resolve persisting login issues.

3. **Report Generation - Ticket T0003**  
   - Action: Enhance the training for agents dealing with report generation.  
   - Suggestion: Develop a fast-track troubleshooting guide specifically for report generation issues to improve first-call resolutions.

4. **Data Import - Ticket T0004**  
   - Action:



# Agent: Report Generator
## Final Answer: 
**Issue Classification Results:**
| Issue Type        | Frequency | Priority Levels |
|-------------------|-----------|------------------|
| API Issue         | 5         | High (3), Medium (2) |
| Login Issue       | 8         | Critical (2), High (5), Low (1) |
| Report Generation  | 6         | High (2), Medium (2), Low (2) |
| Data Import       | 12        | High (5), Medium (5), Low (2) |
| Feature Request   | 8         | Critical (2), High (4), Medium (2) |
| Billing Issue     | 12        | Critical (3), High (5), Medium (4) |
| UI Bug            | 7         | High (3), Medium (3), Critical (1) |

**Agent Performance:**
| Agent ID | Total Resolved | Total Tickets | Avg Resolution Time (mins) | Avg Satisfaction Rating |
|----------|----------------|---------------|-----------------------------|--------------------------|
| A001     | 7              | 12            | 843.33                      | 2.67                     |
| A002     | 



# Agent: Report Generator
## Final Answer: 
**Support Ticket Summary Report**
---

### **1. Issue Classification Results:**
| Issue Type        | Frequency | Priority Levels                 |
|-------------------|-----------|----------------------------------|
| API Issue         | 5         | High (3), Medium (2)            |
| Login Issue       | 8         | Critical (2), High (5), Low (1) |
| Report Generation  | 6         | High (2), Medium (2), Low (2)   |
| Data Import       | 12        | High (5), Medium (5), Low (2)   |
| Feature Request   | 8         | Critical (2), High (4), Medium (2) |
| Billing Issue     | 12        | Critical (3), High (5), Medium (4) |
| UI Bug            | 7         | High (3), Medium (3), Critical (1) |

![Issue Distribution](./issue_distribution.png)  
![Ticket Breakdown by Priority Level](./priority_levels.png)

### **2. Agent Performance:**
| Agent ID | Total Resolved | Total Tickets | Avg Resolution Time (mins) | Avg Satisfaction Rating |
|------

                             Tasks Scores                              
                        (1-10 Higher is better)                        
┏━━━━━━━━━━━━━━━━━━━━┯━━━━━━━┯━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━┯━━┯━━┓
┃ Tasks/Crew/Agents  │ Run 1 │ Avg. Total │ Agents              │  │  ┃
┠────────────────────┼───────┼────────────┼─────────────────────┼──┼──┨
┃ Task 1             │  9.0  │    9.0     │ - Suggestion Engine │  │  ┃
┃                    │       │            │                     │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Task 2             │  8.5  │    8.5     │ - Report Generator  │  │  ┃
┃                    │       │            │                     │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Task 3             │  9.5  │    9.5     │ - Chart Specialist  │  │  ┃
┃                    │       │            │                     │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Task 4             │  8.5  │    8.5     │ - Report Generator  │  │  ┃
┃                    │       │            │                     │  │  ┃
┃ Crew               │ 8.88  │    8.9     │                     │  │  ┃
┃ Execution Time (s) │   7   │     7      │                     │  │  ┃
┗━━━━━━━━━━━━━━━━━━━━┷━━━━━━━┷━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━┷━━┷━━┛

## Kicking off Crew

In [9]:
result = support_report_crew.kickoff()

# Agent: Suggestion Engine
## Task: Generate actionable suggestions for resolving each classified support ticket. The suggestions should be based on: - Issue Type: Tailor suggestions to the specific type of issue reported. - Historical Data: Use historical data such as resolution_time_minutes and
  satisfaction_rating to inform the suggestions.
- Customer Feedback: Incorporate insights from customer_comments to
  customize the suggestions further.

The goal is to provide clear, actionable steps that the support team can take to resolve each issue efficiently and effectively.



# Agent: Suggestion Engine
## Using tool: Read a file's content
## Tool Input: 
"{\"file_path\": \"./support_tickets_data.csv\"}"
## Tool Output: 
ticket_id,customer_id,issue_type,issue_description,priority,date_submitted,response_time_minutes,resolution_time_minutes,satisfaction_rating,customer_comments,agent_id,resolved
T0001,C0511,API Issue,I'm pleased with how my issue was handled. Thanks!,High,2023-03-25,24



# Agent: Suggestion Engine
## Final Answer: 
1. **API Issue - Ticket T0001**  
   - Action: Continue to utilize efficient communication protocols to prevent slowdowns.   
   - Suggestion: Ensure agents follow up on recurring API issues based on previous feedback that hints at frequent occurrences. Improve resolution speed by using pre-defined scripts for common API issues.

2. **Login Issue - Ticket T0002**  
   - Action: Investigate any system outages during the time the customer attempted to log in.  
   - Suggestion: Consistently monitor system status and provide alerts to customers about any ongoing issues. Follow the escalation path to quickly resolve persisting login issues.

3. **Report Generation - Ticket T0003**  
   - Action: Enhance the training for agents dealing with report generation.  
   - Suggestion: Develop a fast-track troubleshooting guide specifically for report generation issues to improve first-call resolutions.

4. **Data Import - Ticket T0004**  
   - Action:



# Agent: Chart Specialist
## Final Answer: 
1. **Issue Distribution**
   - A pie chart showing the distribution of different issue types based on their frequency.
   - Code to generate and save this chart:
   ```python
   import matplotlib.pyplot as plt

   issue_types = ["API Issue", "Login Issue", "Report Generation", "Data Import", "Feature Request", "Billing Issue", "UI Bug"]
   frequencies = [5, 8, 6, 12, 8, 12, 7]

   plt.figure(figsize=(10, 6))
   plt.pie(frequencies, labels=issue_types, autopct='%1.1f%%', startangle=140)
   plt.title("Issue Distribution")
   plt.axis('equal')
   plt.savefig("issue_distribution.png")
   plt.close()
   ```
   - URL: [issue_distribution.png](./issue_distribution.png)

2. **Priority Levels**
   - A bar chart depicting the breakdown of tickets by priority level.
   - Code to generate and save this chart:
   ```python
   import numpy as np

   priorities = ['Critical', 'High', 'Medium', 'Low']
   counts = [5, 22, 17, 7]  # Summarized counts

   plt



# Agent: Report Generator
## Final Answer: 
**Support Ticket Summary Report**
---

### **1. Issue Classification Results:**
| Issue Type        | Frequency | Priority Levels                 |
|-------------------|-----------|----------------------------------|
| API Issue         | 5         | High (3), Medium (2)            |
| Login Issue       | 8         | Critical (2), High (5), Low (1) |
| Report Generation  | 6         | High (2), Medium (2), Low (2)   |
| Data Import       | 12        | High (5), Medium (5), Low (2)   |
| Feature Request   | 8         | Critical (2), High (4), Medium (2) |
| Billing Issue     | 12        | Critical (3), High (5), Medium (4) |
| UI Bug            | 7         | High (3), Medium (3), Critical (1) |

![Issue Distribution](./issue_distribution.png)  
![Ticket Breakdown by Priority Level](./priority_levels.png)

### **2. Agent Performance:**
| Agent ID | Total Resolved | Total Tickets | Avg Resolution Time (mins) | Avg Satisfaction Rating |
|------

## Result

In [10]:
from IPython.display import display, Markdown
display(Markdown(result.raw))

**Support Ticket Summary Report**
---

### **1. Issue Classification Results:**
| Issue Type        | Frequency | Priority Levels                 |
|-------------------|-----------|----------------------------------|
| API Issue         | 5         | High (3), Medium (2)            |
| Login Issue       | 8         | Critical (2), High (5), Low (1) |
| Report Generation  | 6         | High (2), Medium (2), Low (2)   |
| Data Import       | 12        | High (5), Medium (5), Low (2)   |
| Feature Request   | 8         | Critical (2), High (4), Medium (2) |
| Billing Issue     | 12        | Critical (3), High (5), Medium (4) |
| UI Bug            | 7         | High (3), Medium (3), Critical (1) |

![Issue Distribution](./issue_distribution.png)  
![Ticket Breakdown by Priority Level](./priority_levels.png)

### **2. Agent Performance:**
| Agent ID | Total Resolved | Total Tickets | Avg Resolution Time (mins) | Avg Satisfaction Rating |
|----------|----------------|---------------|-----------------------------|--------------------------|
| A001     | 7              | 12            | 843.33                      | 2.67                     |
| A002     | 9              | 10            | 823.00                      | 2.75                     |
| A003     | 10             | 12            | 732.25                      | 2.5                      |
| A004     | 7              | 16            | 659.33                      | 2.75                     |
| A005     | 8              | 10            | 702.88                      | 2.88                     |

![Agent Performance](./agent_performance.png)

### **3. Customer Satisfaction Over Time:**
| Date       | Avg Satisfaction Rating | Total Tickets |
|------------|-------------------------|---------------|
| Jan 2023   | 2.67                    | 11            |
| Feb 2023   | 2.67                    | 10            |
| Mar 2023   | 2.77                    | 14            |
| Apr 2023   | 3.55                    | 14            |
| May 2023   | 2.67                    | 14            |
| Jun 2023   | 3.09                    | 8             |
| Jul 2023   | 3.00                    | 1             |

![Customer Satisfaction Ratings Over Time](./customer_satisfaction.png)

### **4. Suggested Actions:**
- **API Issues**
  - Action: Continue to utilize efficient communication protocols to prevent slowdowns.
  - Suggestion: Ensure agents follow up on recurring API issues based on previous feedback that hints at frequent occurrences.

- **Login Issues**
  - Action: Investigate any system outages during the time the customer attempted to log in.
  - Suggestion: Consistently monitor system status and provide alerts to customers about any ongoing issues.

- **Report Generation**
  - Action: Enhance the training for agents dealing with report generation.
  - Suggestion: Develop a fast-track troubleshooting guide specifically for report generation issues to improve first-call resolutions.

- **Data Import**
  - Action: Recognize the customer’s satisfaction with swift resolution, apply similar processes to unresolved tickets.
  - Suggestion: Document the successful resolution steps taken and identify any additional training for handling similar queries rapidly.

- **Feature Requests**
  - Action: Communicate to the customer that their feedback will be taken into consideration for future improvements.
  - Suggestion: Create a feedback loop where customers know their feature requests are valued and are being actively assessed, addressing their frustrations with follow-ups.

- **Billing Issues**
  - Action: Focus on the customer’s indication of unresolved issues.
  - Suggestion: Increase transparency regarding the billing resolutions and ensure billing queries are quickly prioritized.

- **UI Bugs**
  - Action: Promote knowledge sharing among agents regarding efficient resolution strategies for UI bugs.
  - Suggestion: Develop usability testing protocols to preemptively identify potential UI issues before they reach customers.

---

This report compiles the critical metrics, trends, and actionable insights observed in the support system, providing valuable information to stakeholders for decision making and continuous improvement.
```